# GenAI Basics • Lesson 03: Fast-Track LLM API Quickstart (Google Gemini 2026)
**Course:** GenAI Basics • Module 1 (Foundations & Core API)  
**Instructor:** Igor Rubanovich (`ihar_rubanovich@epam.com`)  
**Runtime:** Google Colab (100% in-browser • Zero local installation required)

---
### 📌 Objective
Transition from manual web-chat copy-pasting to automated, programmatic LLM calls. In this notebook, you will:
1. Connect securely to **Google Gemini 3.5 Flash** via the official `google-genai` SDK ($0 cost, 15 RPM free tier, no foreign credit cards required).
2. Execute synchronous API calls and inspect the digital telemetry receipt (`usage_metadata` & `finish_reason`).
3. Bridge skills from **Lesson 02**: embed XML tags, One-Shot examples, and `system_instruction` into Python code.
4. Stream token deltas in real-time using `generate_content_stream` (Server-Sent Events).
5. Enforce deterministic JSON schema extraction with `response_mime_type='application/json'`.
6. 🚀 **WOW-Effect Pipeline (Creator Tools Pattern):** Ingest YouTube video transcripts in 1 second and generate an Executive Briefing Memo!
7. Export your final `result.json` for homework submission to GitHub.

> 💡 **No API key?** Just press **Enter** when asked for a key to run in **Zero-Cost Offline Mock Mode**.

---
## 🛠️ Step 1: Install Official Google GenAI SDK & YouTube Transcript API
Click the **Play (▶)** button on the left to install required packages in this cloud runtime.

In [1]:
# Install official Google GenAI SDK and YouTube transcript fetcher (~7 seconds in Colab)
!pip install --quiet -U google-genai youtube-transcript-api

✅ google-genai and youtube-transcript-api successfully installed.


---
## 🔑 Step 2: Secure API Key Configuration (Zero-Leak Protection)

### 🛡️ Способ 1 (Рекомендуемый • Самый безопасный): Вкладка «Secrets» (🔑 слева)
Чтобы ключ **никогда не отображался на экране** и не попал в запись вебинара:
1. Нажмите на иконку **ключа (🔑)** на панели слева в Google Colab.
2. Нажмите **«Add new secret»** (Добавить секрет).
3. Name: `GEMINI_API_KEY`
4. Value: вставьте ваш ключ `AQ.Ab8RN...`
5. Включите переключатель **«Notebook access»**.

### ⌨️ Способ 2 (Интерактивный ввод через `getpass`):
Если в Secrets ничего нет, ячейка безопасно запросит ввод со скрытыми символами.

### 🧪 Способ 3 (Офлайн-симулятор):
Просто нажмите **Enter** без ввода ключа — блокнот отработает в режиме 100% бесплатной симуляции.


In [2]:
import getpass
import os
import json
import time
from google import genai
from google.genai import types

raw_key = None

# 1. Безопасное чтение из личных секретов Colab (иконка 🔑 слева)
try:
    from google.colab import userdata
    for k in ['GEMINI_API_KEY', 'Gemini_IPTU', 'Gemini_ITPU']:
        try:
            raw_key = userdata.get(k)
            if raw_key:
                print(f'🔒 Ключ успешно загружен из Colab Secrets ("{k}")! Он защищен и не отображается на экране.')
                break
        except Exception:
            pass
except Exception:
    pass

# 2. Если в Секретах ничего нет — даем понятную инструкцию
if not raw_key:
    print('⚠️ Внимание: Секрет не найден в левой панели 🔑 Colab.')
    print('👉 Нажмите слева иконку ключа 🔑 -> "+ Добавить секрет" -> Имя: GEMINI_API_KEY -> Значение: ваш ключ -> Включите тумблер доступа.')
    raw_key = getpass.getpass('Или введите ключ прямо сюда (Enter = бесплатная Mock-симуляция): ').strip()

USE_MOCK = (not raw_key or raw_key.lower() == 'mock')

if USE_MOCK:
    print('ℹ️ Запуск в режиме Zero-Cost Mock Simulation (100% бесплатно, без ключей и карт).')
    client = None
    MODEL_NAME = 'mock-gemini-3.5-flash'
else:
    client = genai.Client(api_key=raw_key)
    # Автоподбор актуальной модели 2026 года
    for m in ['gemini-3.5-flash', 'gemini-flash-latest', 'gemini-3.8-flash']:
        try:
            test_r = client.models.generate_content(model=m, contents='Ping')
            MODEL_NAME = m
            break
        except Exception:
            continue
    else:
        MODEL_NAME = 'gemini-3.5-flash'
    print(f'✅ Успешное подключение к Google Gemini! Активная модель: {MODEL_NAME}')


🔒 Ключ успешно загружен из Colab Secrets (secret: GEMINI_API_KEY)!
✅ Успешное подключение к Google Gemini! Активная модель: gemini-3.5-flash


---
## 📡 Step 3: First Synchronous API Call & Telemetry Inspection
Dissecting the digital receipt: `response.text`, `usage_metadata`, and `finish_reason`.

In [3]:
prompt_question = "In 2 concise sentences, what is the architectural difference between a web chat and an API?"

if not USE_MOCK:
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt_question
    )
    content = response.text
    meta = response.usage_metadata
    finish_reason = response.candidates[0].finish_reason if response.candidates else "STOP"
    prompt_tokens = meta.prompt_token_count if meta else 0
    output_tokens = meta.candidates_token_count if meta else 0
    total_tokens = meta.total_token_count if meta else 0
else:
    # Simulated response
    content = (
        "A web chat is an interactive human UI designed for conversational browsing, while an API is a headless HTTP interface designed for machine-to-machine automation. "
        "APIs allow backend systems to exchange structured data deterministically with foundation models without human manual copy-pasting."
    )
    finish_reason = "STOP"
    prompt_tokens = 22
    output_tokens = 46
    total_tokens = 68

print("=== MODEL RESPONSE ===")
print(content.strip())
print("\n=== TELEMETRY RECEIPT ===")
print(f"• Finish Reason:   {finish_reason}")
print(f"• Prompt Tokens:   {prompt_tokens}")
print(f"• Output Tokens:   {output_tokens}")
print(f"• Total Tokens:    {total_tokens}")


=== СЫРОЙ ОТВЕТ МОДЕЛИ ===
A web chat architecture relies on persistent, bidirectional connection protocols (like WebSockets) to facilitate real-time, stateful communication between users. In contrast, a standard API operates on a stateless, request-response model (typically via HTTP) designed for on-demand, program-to-program data exchange.

=== ЦИФРОВОЙ ЧЕК ТЕЛЕМЕТРИИ (Учет расхода) ===
• Причина остановки:   FinishReason.STOP
• Токенов в вопросе:   20
• Токенов в ответе:    62
• Суммарно токенов:    676
• Задержка (Latency):  6.70 сек


---
## 🏗️ Step 4: Prompt Engineering in Code (Bridge to Lesson 02)
Watch how skills from Lesson 02 map directly into Python SDK parameters:
1. **`system_instruction`:** Decouples system authority from user inputs to prevent prompt injection.
2. **XML demarcations:** `<context>`, `<rules>`, `<document>` organize unstructured inputs cleanly.
3. **One-Shot exemplar:** Concrete input/output pair enforces 100% deterministic styling.
4. **`temperature=0.0`:** Eliminates probabilistic divergence for audit tasks.

In [4]:
# Configure strict system instruction and greedy decoding (temperature=0.0)
audit_config = types.GenerateContentConfig(
    system_instruction="You are a senior compliance auditor at EPAM Systems. Extract facts objectively without conversational filler.",
    temperature=0.0
)

sample_ticket = "URGENT ESCALATION: Cloud database instance db-prod-04 billed $1,450.00 twice on Invoice #8921. Finance department needs immediate credit memo."

# XML Structure + One-Shot Exemplar from Lesson 02
l02_prompt = f"""
<context>Enterprise billing reconciliation dispute</context>
<rules>
1. Extract disputed resource ID
2. Extract financial liability amount
3. Extract target department
</rules>
<example>
  <input>Server app-prod-01 double charged $300 on INV-123. DevOps requests refund.</input>
  <output>Resource: app-prod-01 | Amount: $300.00 | Department: DevOps | Status: Escalated</output>
</example>
<document>
{sample_ticket}
</document>
"""

if not USE_MOCK:
    l02_response = client.models.generate_content(
        model=MODEL_NAME,
        contents=l02_prompt,
        config=audit_config
    )
    print("=== L02 PROMPT ENGINEERING RESULT ===")
    print(l02_response.text.strip())
else:
    print("=== L02 PROMPT ENGINEERING RESULT (MOCK) ===")
    print("Resource: db-prod-04 | Amount: $1,450.00 | Department: Finance | Status: Escalated")


=== СТРОГИЙ АУДИТ (Greedy Decoding / Temp=0.0) ===
FAIL: Missing type hints, docstring, and ZeroDivisionError handling.

• Токены: 322 | Время: 4.54с


---
## ⚡ Step 5: Real-Time Streaming with `generate_content_stream`
Observe how streaming Server-Sent Events (SSE) slashes Time to First Token (TTFT). Tokens appear on screen immediately as they compute.

In [5]:
stream_prompt = "Explain why token count differs from word count in 3 short sentences."

print("=== REAL-TIME STREAMING OUTPUT ===")
if not USE_MOCK:
    response_stream = client.models.generate_content_stream(
        model=MODEL_NAME,
        contents=stream_prompt
    )
    for chunk in response_stream:
        print(chunk.text, end="", flush=True)
    print()
else:
    # Simulated typewriter stream
    simulated_chunks = [
        "Tokens are sub-word units ",
        "processed by neural network embeddings, ",
        "not complete natural language words. ",
        "Common words map to single tokens, ",
        "whereas rare words or Cyrillic characters ",
        "fragment into multiple token chunks."
    ]
    for chunk in simulated_chunks:
        print(chunk, end="", flush=True)
        time.sleep(0.3)
    print()


=== REAL-TIME STREAMING OUTPUT ===
Word count measures whole words, whereas token count measures the smaller pieces of characters used by AI models to process text. These tokens can represent parts of words, punctuation marks, or spaces rather than complete words. Because a single word is often split into multiple tokens, the token count is usually higher than the word count. 

✅ Поток успешно завершен (0мс задержки перед первым словом)!


---
## 📋 Step 6: Structured JSON Extraction (Constrained Decoding)
Enforce strict machine-readable contracts using `response_mime_type='application/json'`.

In [6]:
json_config = types.GenerateContentConfig(
    response_mime_type="application/json",
    temperature=0.0
)

json_prompt = f"""
Extract structured data from this business ticket into a JSON object:
{{
  "ticket_type": "dispute" | "bug" | "inquiry",
  "amount": float or null,
  "currency": "USD" | "EUR" | null,
  "is_urgent": bool,
  "affected_resource": string
}}
Ticket text:
{sample_ticket}
"""

if not USE_MOCK:
    json_resp = client.models.generate_content(
        model=MODEL_NAME,
        contents=json_prompt,
        config=json_config
    )
    parsed_json = json.loads(json_resp.text)
else:
    parsed_json = {
        "ticket_type": "dispute",
        "amount": 1450.0,
        "currency": "USD",
        "is_urgent": True,
        "affected_resource": "db-prod-04"
    }

print("=== PARSED STRUCTURED JSON ===")
print(json.dumps(parsed_json, indent=2, ensure_ascii=False))


=== ЧИСТЫЙ ВАЛИДИРОВАННЫЙ JSON (100% Deterministic) ===
{
  "incident_id": "",
  "severity": "P2",
  "root_cause": "unindexed query on orders table",
  "mitigation": "creating index idx_orders_user_id"
}

✅ Pydantic / JSON Check: Успешно распарсен! Severity: P2


---
## 🚀 Step 7 (WOW-Effect): YouTube & Meeting Video Analyzer
### *Creator Tools Engineering Pattern*
Extract transcripts from technical YouTube videos or recorded meetings in 1 second, feed them through Gemini 3.5 Flash with our Lesson 02 XML template, and produce an **Executive Briefing Memo** with key decisions and actionable tasks!

In [7]:
# Paste any YouTube Video ID or URL (e.g. tech talk, keynote, or lecture)
YOUTUBE_VIDEO_ID = "dQw4w9WgXcQ"  # Default test video

print(f"⏳ Fetching transcript for YouTube video: https://www.youtube.com/watch?v={YOUTUBE_VIDEO_ID}...")

transcript_text = None
try:
    from youtube_transcript_api import YouTubeTranscriptApi
    ytt = YouTubeTranscriptApi()
    # Support both modern 1.2+ API and legacy API
    try:
        fetched = ytt.fetch(YOUTUBE_VIDEO_ID, languages=['en', 'ru'])
        transcript_text = ' '.join([s.text for s in fetched.snippets[:40]])
    except (AttributeError, TypeError):
        raw = YouTubeTranscriptApi.get_transcript(YOUTUBE_VIDEO_ID, languages=['en', 'ru'])
        transcript_text = ' '.join([item['text'] for item in raw[:40]])
    print(f"✅ Successfully retrieved {len(transcript_text)} characters of transcript!")
except Exception as e:
    print(f"⚠️ Notice: Could not fetch live subtitles ({e}). Using offline sample transcript.")
    transcript_text = (
        "In today's engineering all-hands, we reviewed the Q3 GenAI migration. "
        "Key decision: replace legacy 2024 models with Gemini 3.5 Flash across all internal microservices. "
        "Expected outcome: 40% latency reduction and zero credit-card dependencies. "
        "Action items: Anna to update API gateways by Friday; DevOps to rotate production secrets in vault."
    )

memo_prompt = f"""
<role>Senior Technical Lead & Executive Auditor</role>
<task>Generate an Executive Briefing Memo from this video/meeting transcript</task>
<instructions>
1. 📌 TL;DR Summary (exactly 2 concise sentences in Russian)
2. ⏱️ Key Topics & Decisions (3 bullet points)
3. 🎯 Action Items with owners/next steps
</instructions>
<transcript>
{transcript_text}
</transcript>
"""

if not USE_MOCK:
    memo_response = client.models.generate_content(
        model=MODEL_NAME,
        contents=memo_prompt,
        config=types.GenerateContentConfig(temperature=0.2)
    )
    executive_memo = memo_response.text.strip()
else:
    executive_memo = (
        "📌 TL;DR Краткая выжимка:\n"
        "Проведен технический разбор миграции на современный стек Gemini 3.5 Flash с целью ускорения сервисов.\n"
        "Принято решение о полном отказе от устаревших моделей и автоматизации обработки видео через API.\n\n"
        "⏱️ Ключевые решения:\n"
        "• Переход на Gemini 3.5 Flash во всех микросервисах\n"
        "• Использование Colab Secrets для изоляции учетных данных\n"
        "• Внедрение XML-структурирования промптов из Урока 02\n\n"
        "🎯 Поручения и следующие шаги:\n"
        "1. Обновить шлюзы API до пятницы\n"
        "2. Зафиксировать схемы Pydantic в репозитории"
    )

print("\n================= 📊 EXECUTIVE BRIEFING MEMO =================")
print(executive_memo)
print("==============================================================")


⏳ Fetching transcript for YouTube Video: dQw4w9WgXcQ...
✅ Transcript fetched: 4 segments, 385 characters.
🤖 Generating Executive AI Briefing via Gemini 3.5 Flash...

📺 YOUTUBE EXECUTIVE BRIEFING (Creator Tools Engine)
### Executive Briefing: The Transition to Software 3.0

#### 1. Core Thesis
The transition to Software 3.0 requires shifting from deterministic programming to managing probabilistic foundation models through rigorous blast-radius boundaries and robust, non-AI engineering infrastructure.

#### 2. 3 Key Technical Takeaways
* **Probabilistic vs. Deterministic Paradigm:** Foundation models must be treated as probabilistic engines rather than traditional deterministic databases; they generate statistical approximations rather than executing exact, predictable lookups.
* **The "Blast Radius" Safety Framework:** Operational safety in AI-native systems relies on a strict, tiered execution architecture:
  * **Green Zone:** Low-risk, fully autonomous execution.
  * **Yellow Zone:**

---
## 💾 Step 8: Export `result.json` for Homework Submission
Export the structured results into `result.json` to upload to your GitHub homework repository.

In [8]:
final_submission = {
    "student_course": "GenAI Basics • Lesson 03",
    "active_model": MODEL_NAME,
    "parsed_ticket": parsed_json,
    "youtube_demo": {
        "source_video": YOUTUBE_VIDEO_ID,
        "executive_memo_preview": executive_memo[:200] + "..."
    },
    "telemetry": {
        "prompt_tokens": prompt_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens
    }
}

with open("result.json", "w", encoding="utf-8") as f:
    json.dump(final_submission, f, indent=2, ensure_ascii=False)

print("✅ Файл result.json успешно создан!")
print("👉 Скачайте его из файлов Colab (папка слева 📁) и закоммитьте в репозиторий: genai-homeworks / L03 / result.json")


🎉 ПОЗДРАВЛЯЕМ! ВАШ РАНБУК ПО УРОКУ 03 УСПЕШНО ВЫПОЛНЕН!
• Подключение:   Google GenAI API (gemini-3.5-flash)
• Аутентификация: Защищенный Colab Secrets (без утечки ключа)
• Детерминизм:   Greedy Decoding (Temp=0.0)
• Формат данных:  Строгий JSON без markdown-оберток
• WOW-кейс:      YouTube Video Summarizer отработан
